In [1]:

import os


CCDC_MOGUL_INITIALISATION_FILE='/nfs/home/myuecel3/miniforge3/envs/frontiers_medchem/lib/python3.10/site-packages/ccdc/parameter_files/mogul.ini'
os.environ['CCDC_MOGUL_INITIALISATION_FILE'] = CCDC_MOGUL_INITIALISATION_FILE
from ccdc.docking import Docker
from ccdc.io import MoleculeReader, EntryReader, EntryWriter, MoleculeWriter
from ccdc.protein import Protein
from pathlib import Path
from ccdc.molecule import Molecule


/nfs/home/myuecel3/miniforge3/envs/frontiers_medchem/lib/python3.10/site-packages/ccdc/__init__.py:237: UserWarning: 
Could not find the CSD database in the CSD Software installation.  Data dependent features will not be available.

For further help with installing and configuring data please visit
the support page at https://www.ccdc.cam.ac.uk/csds_install_help

  warnings.warn(CSD_DATABASE_NOT_FOUND_MSG)


In [56]:
MAIN_FOLDER = Path('/nfs/home/myuecel3/Frontiers_MedChem')
DATA_FOLDER = 'data' #Please change this to your own path
protein_file = '/nfs/home/myuecel3/Frontiers_MedChem/data/protein_ligand_4LQM_CHEMBL674017.mol2' #Please change this to your own path
native_ligand_file = '${MAIN_FOLDER}/${DATA_FOLDER}/4lqm_ligand.mol2' #Please change this to your own path

In [57]:
out_folder = DATA_FOLDER
protein = Protein.from_file(protein_file)
protein.remove_all_waters()
protein.remove_unknown_atoms()
protein.add_hydrogens()
for lig in protein.ligands:
    protein.remove_ligand(lig.identifier)
prep_protein_file =  os.path.join(out_folder, 'protein_prepared.mol2')
with EntryWriter(prep_protein_file) as writer:
    writer.write(protein)


OSError: Could not open data/protein_prepared.mol2 for writing.

Binding Site Definition
- From the reference ligand
- From a protein atom, residue(s) or point

In [ ]:
#From the native ligand
native_ligand = MoleculeReader(native_ligand_file)[0]
docker = Docker()

settings = docker.settings
settings.binding_site = settings.BindingSiteFromLigand(protein, native_ligand, 10.0)

atoms = settings.binding_site.atoms
print(f'Number of atoms in the binding site: {len(atoms)}')
residues = settings.binding_site.residues
print(f'Number of residues in the binding site: {len(residues)}')
waters = settings.binding_site.waters
print(f'Number of waters in the binding site: {len(waters)}')
metal_ions = settings.binding_site.metals
print(f'Number of metal ions in the binding site: {len(metal_ions)}')
ligands = settings.binding_site.ligands
print(f'Number of ligands in the binding site: {len(ligands)}')


Number of atoms in the binding site: 802
Number of residues in the binding site: 76
Number of waters in the binding site: 0
Number of metal ions in the binding site: 0
Number of ligands in the binding site: 0


In [34]:
#Alternatively, you can specify the binding site by using an atom or a residue(s). For example;
VAL717 = [r for r in protein.residues if r.identifier == 'A:VAL717'][0]
settings.binding_site = settings.BindingSiteFromResidue(protein, VAL717, 10.0)
atoms = settings.binding_site.atoms
print(f'Number of atoms in the binding site: {len(atoms)}')
residues = settings.binding_site.residues
print(f'Number of residues in the binding site: {len(residues)}')
waters = settings.binding_site.waters
print(f'Number of waters in the binding site: {len(waters)}')
metal_ions = settings.binding_site.metals
print(f'Number of metal ions in the binding site: {len(metal_ions)}')
ligands = settings.binding_site.ligands
print(f'Number of ligands in the binding site: {len(ligands)}')


Number of atoms in the binding site: 353
Number of residues in the binding site: 22
Number of waters in the binding site: 0
Number of metal ions in the binding site: 0
Number of ligands in the binding site: 0


In [ ]:
mol_a = Molecule.from_string("C=CC(=O)Nc1ccc2ncnc(Nc3ccc(F)c(Br)c3)c2c1")
print(type(mol_a))
lig_prep = Docker.LigandPreparation()


<class 'ccdc.molecule.Molecule'>


AttributeError: 'Molecule' object has no attribute 'molecule'

In [59]:
from ccdc import io
from ccdc import conformer
conformer_gen = conformer.ConformerGenerator()
conformer_gen.settings.max_conformers = 10
conformers = conformer_gen.generate(mol_a)



In [ ]:
from ccdc.diagram import DiagramGenerator
DiagramGenerator = DiagramGenerator()
img = DiagramGenerator.image(mol_a)
img

In [78]:
lig = '/nfs/home/myuecel3/Frontiers_MedChem/data/4lqm_ligand.mol2'

lig_prep = Docker.LigandPreparation()
prepared_ligand = lig_prep.prepare(EntryReader(lig)[0])